# 13 - Transformer Block

In the previous notebook, we learned Self-Attention.

Now we will build the main block used inside Transformer models.

A Transformer block contains:

1. Multi-Head Attention #Allows the model to focus on different parts of the sentence simultaneously.Multiple heads learn different relationships at the same time.

2. Residual Connection #Adds the original input back to the output so important information is not lost.Suppose someone edits your essay.Instead of replacing your    essay,they keep your original version and add improvements.That's a residual connection.

3. Layer Normalization #Normalizes the values so training remains stable and faster.

4. Feed Forward Network #A small neural network that processes each word independently after attention.


## Why Transformer Block?

Self-Attention helps words communicate with each other.

But attention alone is not enough.

The model also needs:

- Stable training
- Non-linear transformation
- Deep stacking

That is why Transformer uses LayerNorm, Residual Connections, and Feed Forward Networks.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

In [2]:
batch_size = 2
seq_len = 4
embed_dim = 8

x = torch.randn(batch_size, seq_len, embed_dim)

print(x.shape)

torch.Size([2, 4, 8])


In [3]:
multihead_attention = nn.MultiheadAttention(
    embed_dim=embed_dim,
    num_heads=2,
    batch_first=True
)

attention_output, attention_weights = multihead_attention(x, x, x)

print("Attention Output:", attention_output.shape)
print("Attention Weights:", attention_weights.shape)

Attention Output: torch.Size([2, 4, 8])
Attention Weights: torch.Size([2, 4, 4])


In [4]:
residual_output = x + attention_output

print(residual_output.shape)

torch.Size([2, 4, 8])


In [5]:
layer_norm = nn.LayerNorm(embed_dim)

norm_output = layer_norm(residual_output)

print(norm_output.shape)

torch.Size([2, 4, 8])


In [6]:
feed_forward = nn.Sequential(
    nn.Linear(embed_dim, 32),
    nn.ReLU(),
    nn.Linear(32, embed_dim)
)

ff_output = feed_forward(norm_output)

print(ff_output.shape)

torch.Size([2, 4, 8])


In [7]:
final_output = layer_norm(norm_output + ff_output)

print(final_output.shape)

torch.Size([2, 4, 8])


In [8]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

    def forward(self, x):
        attention_output, _ = self.attention(x, x, x)

        x = self.norm1(x + attention_output)

        ff_output = self.feed_forward(x)

        x = self.norm2(x + ff_output)

        return x

In [9]:
block = TransformerBlock(
    embed_dim=8,
    num_heads=2,
    ff_dim=32
)

output = block(x)

print(output.shape)

torch.Size([2, 4, 8])


## Summary

Today I learned:

- Self-Attention is the core operation.
- Multi-Head Attention allows the model to focus on different relationships.
- Residual Connections help information flow.
- Layer Normalization stabilizes training.
- Feed Forward Network transforms token representations.
- These parts together form a Transformer Block.

Input

↓

Multi-Head Attention
(communicate with other words)

↓

Residual Connection
(keep original information)

↓

Layer Normalization
(stabilize values)

↓

Feed Forward Network
(process each word individually)

↓

Residual Connection
(keep useful information)

↓

Layer Normalization
(stabilize again)

↓

Output